# Building GPT from Scratch

This notebook is a personal reconstruction of Andrej Karpathy's tutorial on building a GPT (Generative Pre-trained Transformer) model from scratch. We'll implement a character-level language model using the transformer architecture, training it on Shakespeare text to generate similar-looking text.

**What we'll build:**
- Character-level tokenizer
- Self-attention mechanism with causal masking
- Multi-head attention
- Feed-forward networks
- Complete transformer decoder blocks
- Text generation with the trained model

**Key concepts:**
- Causal (masked) self-attention for autoregressive generation
- Multi-head attention for learning different representation subspaces
- Residual connections and layer normalization
- Positional embeddings to encode token positions

## Configuration

We define all hyperparameters at the top for easy experimentation.

In [1]:
# Model hyperparameters
batch_size = 64
block_size = 256
learning_rate = 3e-4
n_embed = 384
n_layers = 6
n_heads = 6
dropout = 0.2
max_iters = 5000
eval_interval = 100

# Training settings
device = 'cuda'  # 'cuda', 'mps', or 'cpu'

## Setup: Import Libraries

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import requests

torch.manual_seed(1337)

## Load and Prepare Data

We'll use the Tiny Shakespeare dataset - a collection of Shakespeare's works that's perfect for character-level language modeling.

In [3]:
# Download the Shakespeare dataset
response = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")
text = response.text

print(f"Dataset length: {len(text)} characters")
print(f"\nFirst 1000 characters:")
print(text[:1000])

Dataset length: 1115394 characters

First 1000 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
s

## Build Character-Level Tokenizer

Create a simple character-level vocabulary and encoding/decoding functions.

In [4]:
# Get all unique characters and create mappings
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")
print(f"\nExample encoding:")
print(f"  'hello' -> {encode('hello')}")
print(f"  {encode('hello')} -> '{decode(encode('hello'))}'")

Vocabulary size: 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Example encoding:
  'hello' -> [46, 43, 50, 50, 53]
  [46, 43, 50, 50, 53] -> 'hello'


## Create Train/Val Split

Split the data into training and validation sets (90/10 split).

In [5]:
# Encode entire dataset and split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

Training tokens: 1,003,854
Validation tokens: 111,540


## Data Loading: Batch Generation

Create a function to generate random batches of data. Each batch contains multiple sequences of `block_size` tokens.

In [6]:
def get_batch(split):
    """Generate a batch of data for training or validation."""
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Test batch generation
xb, yb = get_batch('train')
print(f"Input batch shape: {xb.shape}")
print(f"Target batch shape: {yb.shape}")
print(f"\nExample: first sequence")
print(f"Input:  {decode(xb[0].tolist())}")
print(f"Target: {decode(yb[0].tolist())}")

Input batch shape: torch.Size([64, 256])
Target batch shape: torch.Size([64, 256])

Example: first sequence
Input:  
Not Gloucester's death, nor Hereford's banishment
Not Gaunt's rebukes, nor England's private wrongs,
Nor the prevention of poor Bolingbroke
About his marriage, nor my own disgrace,
Have ever made me sour my patient cheek,
Or bend one wrinkle on my soverei
Target: Not Gloucester's death, nor Hereford's banishment
Not Gaunt's rebukes, nor England's private wrongs,
Nor the prevention of poor Bolingbroke
About his marriage, nor my own disgrace,
Have ever made me sour my patient cheek,
Or bend one wrinkle on my sovereig


## Self-Attention Head

Implement a single self-attention head with causal masking. This prevents tokens from attending to future positions.

In [7]:
class Head(nn.Module):
    """Single head of self-attention with causal masking."""
    
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        # Compute attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, T)
        # Apply causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # Apply attention to values
        v = self.value(x)  # (B, T, head_size)
        out = wei @ v      # (B, T, head_size)
        return out

## Multi-Head Attention

Combine multiple attention heads in parallel. Each head can learn different aspects of the relationships between tokens.

In [8]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""
    
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

## Feed-Forward Network

A simple feed-forward network applied to each position independently. This adds non-linearity and processing capacity.

In [9]:
class FeedForward(nn.Module):
    """Simple feed-forward network with ReLU activation."""
    
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

## Transformer Block

A complete transformer decoder block with:
1. Multi-head self-attention
2. Feed-forward network
3. Residual connections and layer normalization

In [10]:
class Block(nn.Module):
    """Transformer decoder block: communication followed by computation."""
    
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## Complete GPT Model

Assemble all components into a complete language model with:
- Token and position embeddings
- Stack of transformer blocks
- Final linear layer for token prediction
- Text generation capability

In [11]:
class GPTLanguageModel(nn.Module):
    """Complete GPT language model."""
    
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Embeddings
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb  # (B, T, C)
        
        # Transformer blocks
        x = self.blocks(x)  # (B, T, C)
        x = self.ln_f(x)    # (B, T, C)
        
        # Language modeling head
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        """Generate new tokens autoregressively."""
        for _ in range(max_new_tokens):
            # Crop context to block_size
            idx_cond = idx[:, -block_size:]
            # Get predictions
            logits, _ = self(idx_cond)
            # Focus only on last time step
            logits = logits[:, -1, :]  # (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, C)
            # Sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # Append to sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

## Initialize Model

Create the model and move it to the appropriate device.

In [12]:
# Create model
model = GPTLanguageModel(vocab_size)
model = model.to(device)

# Count parameters
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test generation before training
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(f"\nGeneration before training:")
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))

Model parameters: 10,788,929

Generation before training:

SaVxul&f&&;gXYUPZo
TDuQigFqlHHkPkBj!'zTSbmrDyEJi.?qEU.nIO.h&MHKRP;
qkjEe n&Re!Pp$!DAyC?Dbb$Ib&KIGEfr


## Training Setup

Set up the optimizer and define the loss estimation function.

In [13]:
# Create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    """Estimate loss on train and val sets."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## Training Loop

Train the model for the specified number of iterations.

In [14]:
# Training loop
for iter in range(max_iters):
    # Evaluate loss periodically
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"Step {iter:5d} | Train loss: {losses['train']:.4f} | Val loss: {losses['val']:.4f}")
    
    # Sample a batch
    xb, yb = get_batch('train')
    
    # Forward pass
    logits, loss = model(xb, yb)
    
    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Final loss
losses = estimate_loss()
print(f"\nFinal losses:")
print(f"  Train: {losses['train']:.4f}")
print(f"  Val:   {losses['val']:.4f}")

Step     0 | Train loss: 4.3360 | Val loss: 4.3324
Step   100 | Train loss: 2.4743 | Val loss: 2.4915
Step   200 | Train loss: 2.4148 | Val loss: 2.4455
Step   300 | Train loss: 2.3183 | Val loss: 2.3512
Step   400 | Train loss: 2.1406 | Val loss: 2.1957
Step   500 | Train loss: 2.0093 | Val loss: 2.0910
Step   600 | Train loss: 1.8985 | Val loss: 2.0145
Step   700 | Train loss: 1.8042 | Val loss: 1.9397
Step   800 | Train loss: 1.7262 | Val loss: 1.8752
Step   900 | Train loss: 1.6665 | Val loss: 1.8319
Step  1000 | Train loss: 1.6125 | Val loss: 1.7845
Step  1100 | Train loss: 1.5661 | Val loss: 1.7426
Step  1200 | Train loss: 1.5339 | Val loss: 1.7212
Step  1300 | Train loss: 1.4985 | Val loss: 1.6914
Step  1400 | Train loss: 1.4721 | Val loss: 1.6713
Step  1500 | Train loss: 1.4471 | Val loss: 1.6537
Step  1600 | Train loss: 1.4250 | Val loss: 1.6346
Step  1700 | Train loss: 1.3990 | Val loss: 1.6105
Step  1800 | Train loss: 1.3853 | Val loss: 1.6064
Step  1900 | Train loss: 1.3605

## Generate Text

Use the trained model to generate new Shakespeare-like text.

In [15]:
# Generate text
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_text)


Axes to answer that is 'gainst you not do,
To see you and take I mean, as I heard it;
And bid you by your my nurse. This is your mistress
Do even you atter, as you be thus ballow'd:
So can thou must need?

POMPEY:
So, noble tyranny-son our temple,
Proof, sweet.

ESCALUS:
he was a man could early in work:
I saw the heart Camillo. We are their winds,
They'll give him to eyes: but when this ear is
A capta's. He cannot, we can treast your majesty.

PARIS:
Some little obeyies,
As: they this show stor


## Key Takeaways

**Architecture Components:**
- **Self-attention**: Allows tokens to communicate and gather information from context
- **Multi-head attention**: Multiple attention mechanisms in parallel for richer representations
- **Feed-forward networks**: Per-token computation to process aggregated information
- **Residual connections**: Enable deep networks by providing gradient flow
- **Layer normalization**: Stabilizes training by normalizing activations

**Causal Masking:**
- Prevents tokens from attending to future positions
- Essential for autoregressive generation
- Implemented via triangular mask in attention weights

**Design Choices:**
- Character-level tokenization (simpler but less efficient than BPE)
- Learned positional embeddings (vs. sinusoidal)
- Pre-norm architecture (layer norm before attention/FFN)
- AdamW optimizer with gradient clipping